In [1]:
import os
mykey = os.getenv("OPENAI_APIKEY")
os.environ["USER_AGENT"] = "Chrome/91.0.4472.124"

from langchain_community.document_loaders import WebBaseLoader
try:
    loader=WebBaseLoader(web_paths=["https://www.coursera.org/explore/generative-ai"])
except Exception as e:
    print(f"Error occurred: {e}")
    exit
docs = loader.load()
print (docs)

c:\Users\DELL\Documents\DevjaniDocs\RAG\dev_env\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


[Document(metadata={'source': 'https://www.coursera.org/explore/generative-ai', 'title': 'Explore 300+ AI courses | Coursera', 'description': 'Coursera offers over 300 AI courses taught by experts at Google, Microsoft, IBM, Vanderbilt University, and more.', 'language': 'en'}, page_content="Explore 300+ AI courses | Coursera\n\n\n\n\n\nFor IndividualsFor BusinessesFor UniversitiesFor GovernmentsExploreDegrees\u200bLog InJoin for FreeJoin for Free\n\nGrow essential AI skills and saveGet started by learning the basics or growing your understanding with cutting-edge courses and resume-boosting certificates from AI experts and industry leaders—while unlocking savings on Coursera Plus.Save nowGrow essential AI skills and saveGet started by learning the basics or growing your understanding with cutting-edge courses and resume-boosting certificates from AI experts and industry leaders—while unlocking savings on Coursera Plus.Save nowGrow essential AI skills and saveGet started by learning the

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print (splits[1].page_content)
print (len(splits)) 


Grow essential AI skills and saveGet started by learning the basics or growing your understanding with cutting-edge courses and resume-boosting certificates from AI experts and industry leaders—while unlocking savings on Coursera Plus.Save nowGrow essential AI skills and saveGet started by learning the basics or growing your understanding with cutting-edge courses and resume-boosting certificates from AI experts and industry leaders—while unlocking savings on Coursera Plus.Save nowGrow essential AI skills and saveGet started by learning the basics or growing your understanding with cutting-edge courses and resume-boosting certificates from AI experts and industry leaders—while unlocking savings on Coursera Plus.Save now91% of GenAI learnersachieved a positive career outcome42% of GenAI learners reported a salary increase since enrollment84% of Coursera Plus learnersgained technical skills relevant to their respective industries
11


In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma   
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=mykey)
vectorstore = Chroma.from_documents(splits, embeddings, collection_name="Generative")
print(vectorstore._collection.get())


{'ids': ['a65ec88a-f2b4-406f-a51c-6c90fc6c4649', '11a8c3aa-be3c-45b1-a086-b5336325e24f', 'f7964ac0-1860-48c2-b4dd-d5f3d804ee21', '0ab4afc8-aefe-4801-8ecd-d773ad8303cd', '4aedcdcb-e327-4c8d-8869-2732020083a8', 'c416ec56-bafc-4e67-afb5-2c544a1a55f8', 'be1ce219-b022-4780-acc2-387ea6c6341a', 'fef3f3b3-182e-420d-8144-e59f38de5795', 'e178303b-f3d4-494b-9b0b-11239ad2acbc', '25e85a49-3c16-489d-ab47-3b02b8fd1173', '4704ca71-3556-43b1-8c00-dd72f7e201ac'], 'embeddings': None, 'documents': ['Explore 300+ AI courses | Coursera\n\n\n\n\n\nFor IndividualsFor BusinessesFor UniversitiesFor GovernmentsExploreDegrees\u200bLog InJoin for FreeJoin for Free', 'Grow essential AI skills and saveGet started by learning the basics or growing your understanding with cutting-edge courses and resume-boosting certificates from AI experts and industry leaders—while unlocking savings on Coursera Plus.Save nowGrow essential AI skills and saveGet started by learning the basics or growing your understanding with cutting

In [4]:
retreiver=vectorstore.as_retriever()
from langchain_classic import hub as prompts

prompt= prompts.pull("rlm/rag-prompt")


In [5]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-3.5-turbo", openai_api_key=mykey, temperature=0.9)

In [6]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [7]:
def formatdoc(docs):
    return "\n".join([doc.page_content for doc in docs])    


In [8]:
rag_chain=({"context": retreiver |formatdoc, "question" : RunnablePassthrough()}
           | prompt
           | llm
           | StrOutputParser())

In [11]:
rag_chain.invoke("list the courses available in coursera on generative AI")

'The courses available on Coursera for generative AI include "Accelerate Your Job Search with AI Specialization," "Introduction to Generative AI Learning Path," and "Generative AI for Executives and Business Leaders Specialization."'